Lab 1.1 — Building the Agentic Loop

In [1]:
import sys, subprocess, os
from pathlib import Path

try:
    import anthropic
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "anthropic"], check=True)
    import anthropic

if not os.environ.get("ANTHROPIC_API_KEY"):
    env_path = Path.cwd() / ".env"
    if env_path.exists():
        for line in env_path.read_text().splitlines():
            line = line.strip()
            if not line or line.startswith("#") or "=" not in line:
                continue
            key, _, value = line.partition("=")
            key, value = key.strip(), value.strip()
            if value.startswith("{") and value.endswith("}"):
                value = value[1:-1].strip()
            value = value.strip('"').strip("'")
            if key:
                os.environ.setdefault(key, value)

assert os.environ.get("ANTHROPIC_API_KEY"), "ANTHROPIC_API_KEY is not set."

## Exercise 1 — The Agentic Loop (S1)

Test ticket (used across all exercises):

> From: sarah.chen@globalcorp.com
> Subject: Cannot access SSO login — entire team locked out
> Our team of 40 has been unable to log in via SSO since 09:00 this morning. We have a client demo in 3 hours. This is completely blocking us.

In [2]:
# tools.py
import random

PRODUCT_AREAS = ["Billing", "Platform", "Integrations", "Security", "Onboarding"]
SEVERITIES = ["P1-Critical", "P2-High", "P3-Medium", "P4-Low"]
INTENTS = ["Bug", "Question", "Feature Request", "Billing Dispute"]


def classify_ticket(ticket_text: str, fields_needed: list) -> dict:
    values = {
        "product_area": random.choice(PRODUCT_AREAS),
        "severity": random.choice(SEVERITIES),
        "intent": random.choice(INTENTS),
    }
    return {field: values[field] for field in fields_needed if field in values}

In [3]:
# loop.py
import json
from anthropic import Anthropic

client = Anthropic()
MODEL = "claude-haiku-4-5-20251001"

TICKET_TEXT = (
    "From: sarah.chen@globalcorp.com\n"
    "Subject: Cannot access SSO login — entire team locked out\n\n"
    "Our team of 40 has been unable to log in via SSO since 09:00 this morning. "
    "We have a client demo in 3 hours. This is completely blocking us."
)

tools = [
    {
        "name": "classify_ticket",
        "description": "Classify a support ticket and return values for the requested fields.",
        "input_schema": {
            "type": "object",
            "properties": {
                "ticket_text": {
                    "type": "string",
                    "description": "The full raw text of the support ticket to classify.",
                },
                "fields_needed": {
                    "type": "array",
                    "items": {"type": "string"},
                    "description": (
                        "The classification fields still required, e.g. "
                        "['product_area', 'severity', 'intent']."
                    ),
                },
            },
            "required": ["ticket_text", "fields_needed"],
        },
    }
]

messages = [
    {
        "role": "user",
        "content": (
            "Classify the following support ticket fully. You need three fields: "
            "product_area, severity, and intent. Call the classify_ticket tool as many "
            "times as needed until all three fields are confirmed, then summarize the "
            "final classification in your response.\n\n"
            f"Ticket:\n{TICKET_TEXT}"
        ),
    }
]

iteration = 0
while True:
    iteration += 1
    response = client.messages.create(
        model=MODEL,
        max_tokens=1024,
        tools=tools,
        messages=messages,
    )
    print(f"[iteration {iteration}] stop_reason={response.stop_reason}")

    # Mandatory: append the assistant turn BEFORE any branching.
    messages.append({"role": "assistant", "content": response.content})

    if response.stop_reason == "end_turn":
        final_text = "".join(block.text for block in response.content if block.type == "text")
        print("\nFinal output:\n", final_text)
        break

    if response.stop_reason == "tool_use":
        tool_results = []
        for block in response.content:
            if block.type == "tool_use":
                if block.name == "classify_ticket":
                    result = classify_ticket(**block.input)
                else:
                    result = {"error": f"unknown tool {block.name}"}
                print(f"  tool_use: {block.name}({block.input}) -> {result}")
                tool_results.append(
                    {
                        "type": "tool_result",
                        "tool_use_id": block.id,
                        "content": json.dumps(result),
                    }
                )
        messages.append({"role": "user", "content": tool_results})
        continue

    print(f"Unhandled stop_reason: {response.stop_reason}")
    break

[iteration 1] stop_reason=tool_use
  tool_use: classify_ticket({'ticket_text': 'From: sarah.chen@globalcorp.com\nSubject: Cannot access SSO login — entire team locked out\n\nOur team of 40 has been unable to log in via SSO since 09:00 this morning. We have a client demo in 3 hours. This is completely blocking us.', 'fields_needed': ['product_area', 'severity', 'intent']}) -> {'product_area': 'Security', 'severity': 'P3-Medium', 'intent': 'Bug'}
[iteration 2] stop_reason=end_turn

Final output:
 Perfect! I've successfully classified the support ticket. Here's the final classification summary:

**Ticket Classification:**
- **Product Area:** Security
- **Severity:** P3-Medium
- **Intent:** Bug

**Summary:** This is a security-related bug report of medium severity. A team of 40 users is completely unable to access the system via SSO, which is blocking their work and there's time pressure with a client demo scheduled in 3 hours. While the impact is significant, it was classified as P3-Mediu

### Reflection Questions

- **Tool calls:** Usually 1 (all three fields requested at once); can vary if Claude requests fields incrementally.
- **Tool results before assistant turn:** API returns a 400 error — a `tool_result` must follow the `assistant` turn containing the matching `tool_use` block.
- **`for i in range(2)` instead of `while True`:** The loop can exit before `end_turn`, mid-classification. Iteration counts are not a valid exit condition — only `stop_reason` is.

## Exercise 2 — Coordinator & Subagents (S2)

In [4]:
# subagents.py
import json
from anthropic import Anthropic

client = Anthropic()
MODEL = "claude-haiku-4-5-20251001"


def _strip_code_fences(text: str) -> str:
    text = text.strip()
    if text.startswith("```"):
        text = text.split("\n", 1)[1] if "\n" in text else ""
        if text.endswith("```"):
            text = text[:-3]
    return text.strip()


def run_classifier(ticket: str) -> dict:
    response = client.messages.create(
        model=MODEL,
        max_tokens=512,
        system=(
            "Classify the support ticket into product_area, severity, and intent. "
            "Respond only with a JSON object containing exactly those three keys — "
            "no prose, no markdown code fences."
        ),
        messages=[{"role": "user", "content": ticket}],
    )
    raw_text = "".join(b.text for b in response.content if b.type == "text")
    return json.loads(_strip_code_fences(raw_text))


def run_crm_enricher(customer_email: str, classification: dict) -> dict:
    response = client.messages.create(
        model=MODEL,
        max_tokens=512,
        system=(
            "Simulate a CRM lookup for the given customer email and ticket classification. "
            "Respond only with a JSON object containing exactly these keys: account_tier, "
            "sla_tier, account_manager, contract_value."
        ),
        messages=[
            {
                "role": "user",
                "content": (
                    f"Customer email: {customer_email}\n"
                    f"Classification: {json.dumps(classification)}"
                ),
            }
        ],
    )
    raw_text = "".join(b.text for b in response.content if b.type == "text")
    return json.loads(_strip_code_fences(raw_text))


def run_drafter(ticket: str, classification: dict, crm: dict) -> str:
    context = (
        f"Ticket:\n{ticket}\n\n"
        f"Classification: {json.dumps(classification)}\n\n"
        f"CRM data: {json.dumps(crm)}"
    )
    response = client.messages.create(
        model=MODEL,
        max_tokens=512,
        system=(
            "Draft a professional first-response email to the customer. Reference their "
            "SLA tier explicitly and address the specific product area and severity from "
            "the classification."
        ),
        messages=[{"role": "user", "content": context}],
    )
    return "".join(b.text for b in response.content if b.type == "text")


def run_validator(draft: str, classification: dict, crm: dict) -> str:
    context = (
        f"Draft response:\n{draft}\n\n"
        f"Classification: {json.dumps(classification)}\n\n"
        f"CRM contract — account_tier: {crm.get('account_tier')}, "
        f"sla_tier: {crm.get('sla_tier')}"
    )
    response = client.messages.create(
        model=MODEL,
        max_tokens=512,
        system=(
            "Check the draft against the classification and CRM contract: does it reference "
            "the correct product area, match the customer's SLA/account tier, and meet a "
            "professional SLA tone? Reply with exactly 'APPROVED' if all checks pass, "
            "otherwise list the specific issues."
        ),
        messages=[{"role": "user", "content": context}],
    )
    return "".join(b.text for b in response.content if b.type == "text")

In [5]:
# coordinator.py
CUSTOMER_EMAIL = "sarah.chen@globalcorp.com"

classification = run_classifier(TICKET_TEXT)
print("[Classifier]", classification)

crm = run_crm_enricher(CUSTOMER_EMAIL, classification)
print("[CRM Enricher]", crm)

draft = run_drafter(TICKET_TEXT, classification, crm)
print("[Drafter]\n", draft)

verdict = run_validator(draft, classification, crm)
print("[Validator]", verdict)

[Classifier] {'product_area': 'authentication', 'severity': 'critical', 'intent': 'problem_report'}
[CRM Enricher] {'account_tier': 'enterprise', 'sla_tier': 'premium', 'account_manager': 'Michael Torres', 'contract_value': 250000}
[Drafter]
 **Subject: RE: Cannot access SSO login — entire team locked out [URGENT - Ticket #AUTO-ASSIGNED]**

---

Dear Sarah,

Thank you for reporting this issue. We understand the urgency—a complete SSO outage affecting your entire team is a critical matter that we're treating with the highest priority.

**Your SLA Status:**
As an Enterprise customer on our Premium SLA tier, you are guaranteed a **15-minute first response** and **1-hour resolution target** for critical authentication issues. You have our commitment to address this immediately.

**What We're Doing Now:**
Our senior engineering team in the Authentication product area has been notified and is actively investigating the root cause of the SSO login failure. Given your 3-hour timeline for the c

### Memory Isolation Experiment

Calling `run_drafter(TICKET_TEXT, {}, {})` (no classification/CRM) produces a draft that no longer reliably references the correct product area or SLA tier — the failure mode explicit context passing (Ex 3) prevents.

### Reflection Questions

- **Passing the full Ex 1 `messages` list instead of structured results:** Possible, but costs more tokens and increases hallucination risk since the subagent must re-derive fields from unstructured history.
- **Validator's `stop_reason`:** Always `end_turn` — it has no `tools` list, so no tool-dispatch loop is needed in the caller.

## Exercise 3 — Explicit Context Passing (S3)

In [6]:
# context.py
from dataclasses import dataclass
from typing import Optional


@dataclass
class TicketContext:
    ticket_id: str
    raw_ticket: str
    customer_email: str

    product_area: Optional[str] = None
    severity: Optional[str] = None
    intent: Optional[str] = None

    account_tier: Optional[str] = None
    sla_tier: Optional[str] = None
    account_manager: Optional[str] = None

    draft_response: Optional[str] = None
    validation_result: Optional[str] = None

    def classification_complete(self) -> bool:
        return all(v is not None for v in (self.product_area, self.severity, self.intent))

    def enrichment_complete(self) -> bool:
        return all(v is not None for v in (self.account_tier, self.sla_tier))

    def draft_complete(self) -> bool:
        return self.draft_response is not None


try:
    TicketContext(ticket_id="TCK-2")  # missing raw_ticket, customer_email
except TypeError as e:
    print("Missing-field TypeError raised as expected:", e)

Missing-field TypeError raised as expected: TicketContext.__init__() missing 2 required positional arguments: 'raw_ticket' and 'customer_email'


In [7]:
# coordinator_v2.py
ctx = TicketContext(
    ticket_id="TCK-1001",
    raw_ticket=TICKET_TEXT,
    customer_email="sarah.chen@globalcorp.com",
)

classification = run_classifier(ctx.raw_ticket)
ctx.product_area = classification.get("product_area")
ctx.severity = classification.get("severity")
ctx.intent = classification.get("intent")
print("[Classifier]", classification)

classification_fields = {
    "product_area": ctx.product_area,
    "severity": ctx.severity,
    "intent": ctx.intent,
}
crm = run_crm_enricher(ctx.customer_email, classification_fields)
ctx.account_tier = crm.get("account_tier")
ctx.sla_tier = crm.get("sla_tier")
ctx.account_manager = crm.get("account_manager")
print("[CRM Enricher]", crm)

crm_fields = {
    "account_tier": ctx.account_tier,
    "sla_tier": ctx.sla_tier,
    "account_manager": ctx.account_manager,
}
draft = run_drafter(ctx.raw_ticket, classification_fields, crm_fields)
ctx.draft_response = draft
print("[Drafter]\n", draft)

verdict = run_validator(ctx.draft_response, classification_fields, crm_fields)
ctx.validation_result = verdict
print("[Validator]", verdict)

print("\nFinal context:\n", ctx)

[Classifier] {'product_area': 'authentication', 'severity': 'critical', 'intent': 'support'}
[CRM Enricher] {'account_tier': 'enterprise', 'sla_tier': 'premium', 'account_manager': 'Michael Torres', 'contract_value': '$500000'}
[Drafter]
 **Subject: RE: Cannot access SSO login — entire team locked out [URGENT - TICKET #xxxxx]**

---

Dear Sarah,

Thank you for reporting this issue. We understand the urgency of your situation and have immediately escalated your case as a **Critical** priority ticket.

**Ticket Details:**
- **Product Area:** Authentication (SSO)
- **Severity Level:** Critical
- **Your SLA Tier:** Premium Enterprise
- **Our Response Commitment:** 15-minute initial response, 1-hour resolution target

**What we're doing right now:**
Our senior engineering team is actively investigating the SSO authentication service to identify the root cause of the access blockage affecting your team of 40 users. Given your client demo in 3 hours, this is our top priority.

**Next Steps:**

### Reflection Questions

- **Dict `KeyError` vs. dataclass `TypeError`:** A dict's missing key surfaces later, wherever it's first read. The dataclass fails immediately at construction, naming the exact field — safer for an unattended pipeline.
- **Use of `*_complete()` helpers in Ex 4:** Each becomes the precondition a gate checks before allowing the next subagent call.

## Exercise 4 — Programmatic Step Enforcement (S4)

In [8]:
# gates.py
class PipelineGateError(Exception):
    pass


def gate_classification(ctx):
    if not ctx.classification_complete():
        missing = [
            name
            for name, value in (
                ("product_area", ctx.product_area),
                ("severity", ctx.severity),
                ("intent", ctx.intent),
            )
            if value is None
        ]
        raise PipelineGateError(
            f"Gate 1 (classification) failed — missing fields: {missing}. "
            "Rerun the Classifier before proceeding to CRM Enrichment."
        )


def gate_enrichment(ctx):
    if not ctx.enrichment_complete():
        missing = [
            name
            for name, value in (
                ("account_tier", ctx.account_tier),
                ("sla_tier", ctx.sla_tier),
            )
            if value is None
        ]
        raise PipelineGateError(
            f"Gate 2 (enrichment) failed — missing fields: {missing}. "
            "Rerun the CRM Enricher before proceeding to Drafting."
        )


def gate_draft(ctx):
    if not ctx.draft_complete():
        raise PipelineGateError(
            "Gate 3 (draft) failed — draft_response is None. "
            "Rerun the Drafter before proceeding to Validation."
        )

In [9]:
# coordinator_v3.py
def run_coordinator_v3():
    ctx = TicketContext(
        ticket_id="TCK-1001",
        raw_ticket=TICKET_TEXT,
        customer_email="sarah.chen@globalcorp.com",
    )

    try:
        classification = run_classifier(ctx.raw_ticket)
        ctx.product_area = classification.get("product_area")
        ctx.severity = classification.get("severity")
        ctx.intent = classification.get("intent")
        print("[Classifier]", classification)

        gate_classification(ctx)
        print("Gate 1 passed")

        classification_fields = {
            "product_area": ctx.product_area,
            "severity": ctx.severity,
            "intent": ctx.intent,
        }
        crm = run_crm_enricher(ctx.customer_email, classification_fields)
        ctx.account_tier = crm.get("account_tier")
        ctx.sla_tier = crm.get("sla_tier")
        ctx.account_manager = crm.get("account_manager")
        print("[CRM Enricher]", crm)

        gate_enrichment(ctx)
        print("Gate 2 passed")

        crm_fields = {
            "account_tier": ctx.account_tier,
            "sla_tier": ctx.sla_tier,
            "account_manager": ctx.account_manager,
        }
        draft = run_drafter(ctx.raw_ticket, classification_fields, crm_fields)
        ctx.draft_response = draft
        print("[Drafter]\n", draft)

        gate_draft(ctx)
        print("Gate 3 passed")

        verdict = run_validator(ctx.draft_response, classification_fields, crm_fields)
        ctx.validation_result = verdict
        print("[Validator]", verdict)

        print("\nFinal context:\n", ctx)

    except PipelineGateError as e:
        print(f"[PIPELINE BLOCKED] {e}")


run_coordinator_v3()

[Classifier] {'product_area': 'authentication', 'severity': 'critical', 'intent': 'support_request'}
Gate 1 passed
[CRM Enricher] {'account_tier': 'enterprise', 'sla_tier': 'premium', 'account_manager': 'Michael Torres', 'contract_value': 250000}
Gate 2 passed
[Drafter]
 **Subject: RE: Cannot access SSO login — entire team locked out [URGENT - Ticket #ENG-PRIORITY]**

---

Dear Sarah,

Thank you for reporting this issue. I understand the urgency—a complete SSO authentication outage affecting 40 users with a client demo in 3 hours is critical, and I'm treating this with the highest priority.

**Your Support Details:**
- **SLA Tier:** Premium
- **Severity:** Critical
- **Assigned to:** Myself + escalation team
- **Your Account Manager:** Michael Torres is also being notified

**Immediate Next Steps:**
1. Our authentication engineering team is being engaged right now
2. I'm investigating the SSO service status and any recent deployments
3. You'll receive an update within **15 minutes** wi

### Step 3 — Prove the gate blocks (`coordinator_v3_sabotage.py`)

In [10]:
# coordinator_v3_sabotage.py
def run_coordinator_v3_sabotage():
    ctx = TicketContext(
        ticket_id="TCK-1001",
        raw_ticket=TICKET_TEXT,
        customer_email="sarah.chen@globalcorp.com",
    )

    try:
        classification = run_classifier(ctx.raw_ticket)
        ctx.product_area = classification.get("product_area")
        ctx.severity = classification.get("severity")
        ctx.intent = classification.get("intent")
        print("[Classifier]", classification)

        ctx.severity = None  # sabotage: simulate a partial/dropped classification result

        gate_classification(ctx)
        print("Gate 1 passed")  # never reached

        # Steps 2, 3, 4 never execute once Gate 1 raises.

    except PipelineGateError as e:
        print(f"[PIPELINE BLOCKED] {e}")


run_coordinator_v3_sabotage()

[Classifier] {'product_area': 'authentication', 'severity': 'critical', 'intent': 'bug_report'}
[PIPELINE BLOCKED] Gate 1 (classification) failed — missing fields: ['severity']. Rerun the Classifier before proceeding to CRM Enrichment.


### Reflection Questions

- **Named exception vs. bare `assert`:** `assert` can be stripped entirely with Python's `-O` flag; a named `PipelineGateError` always fires and names the exact missing fields.
- **Auto-retry vs. alert a human:** Transient failures (timeout, malformed JSON) can retry once; systemic failures (invalid field values, repeated `None`) should alert a human.
- **Gate 2 with partial CRM data:** Should still fail — the Validator needs both `account_tier` and `sla_tier` to check the SLA contract. `gate_enrichment` already requires both and names whichever is missing.

## Debrief & Self-Check (§5.1)

1. **stop_reason values:** `tool_use` → execute tool, append result, loop again. `end_turn` → extract text, break. `max_tokens` → log warning, consider retrying with summarized history. `stop_sequence` → treat as `end_turn` unless custom handling is needed.
2. **Drafter's inputs:** Raw ticket text, classification dict, CRM dict — not the full coordinator history, since subagents share no memory and full history wastes tokens while inviting hallucination.
3. **TypeError vs. hallucination:** The TypeError is immediate and points at the exact field, before any API call. A hallucination from missing context is silent and plausible-looking, and may never get caught.
4. **Prompt rule vs. gate:** A prompt rule is advice the model can deprioritize; a gate is Python code that always runs and raises a named error on failure. Use both — the prompt reduces how often the gate fires.
5. **Messages array after two tool calls:** Roles alternate `user, assistant, user, assistant, user` (5 messages) before the model's closing `end_turn` turn is appended (6th).